# Notebook 1

Primo tentativo di affrontare il dataset.
Per affrontare il problema è stata scelta una CNN (Convoluted neural network, classica scelta per immagini).
Simile a architetture viste a lezione si basa su 4 hidden layer convoluzionali secondo il pattern (preso da esempio da ResNet)
Conv -> ReLU -> BN (batch normalization) seguito da una "head" fully connected di 256 -> dropout (0.5) -> 5 (classi del dataset).

Le callback di training sono quelle viste a lezione tra cui:
- early stopping su validation accuracy
- lr adjustment

# Imports

In [ ]:
from keras.layers import BatchNormalization
from keras.layers import Conv2D
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import GlobalAveragePooling2D
from keras.layers import MaxPooling2D
from keras.layers import Rescaling
from keras.models import Sequential

import pathlib
import keras
import tensorflow as tf

## Download dataset

In [ ]:

DATASET_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATASET_NAME = "flower_photos"
CACHE_DIR = "."

data_dir = pathlib.Path(
    tf.keras.utils.get_file(
        origin    = DATASET_URL,
        fname     = DATASET_NAME,
        cache_dir = CACHE_DIR,
        untar     = True,
        extract   = True,
    )
)

### Parameters

In [ ]:
IMG_HEIGHT = 180
IMG_WIDTH  = 180
BATCH_SIZE = 32
SEED       = 67

TRAIN_SPLIT = 0.75
VAL_SPLIT  = 0.15
AUTOTUNE = tf.data.AUTOTUNE

Load the downloaded dataset and split it into train, validation and test sets

In [ ]:
full_ds = keras.utils.image_dataset_from_directory(
  data_dir / DATASET_NAME,
  seed=SEED,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE)

total_batches = len(full_ds)

train_size = int(TRAIN_SPLIT * total_batches)
val_size   = int(VAL_SPLIT * total_batches)

train_ds = full_ds.take(train_size)
val_ds   = full_ds.skip(train_size).take(val_size)
test_ds  = full_ds.skip(train_size + val_size)

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

### Layers

In [ ]:
model = Sequential()
model.add(Rescaling(1./255))

model.add(Conv2D(32, 5, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(64, 3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(128, 3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(256, 3, activation='relu'))
model.add(BatchNormalization())
model.add(GlobalAveragePooling2D())

model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(5))

model.compile(
    optimizer=keras.optimizers.AdamW(1e-4, 1e-4),
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'])

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=f"models/best_checkpoint_test.keras",
        monitor='val_accuracy',
        mode='max',
        save_best_only=True)
]

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

In [ ]:
loss, accuracy = model.evaluate(test_ds)
print(f"Ensemble Accuracy: {accuracy:.4f}")